In [25]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.7 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.6 MB/s eta 0:00:00


In [16]:
import torch
import torch.nn as nn
import pickle
import medmnist
from medmnist import INFO
from torch.utils.data import Dataset, ConcatDataset
from torchvision import transforms, models
from PIL import Image

In [4]:
with open("/kaggle/input/metadata/navigator_metadata.pkl", "rb") as f:
    meta = pickle.load(f)

ANATOMY_LABELS = meta["ANATOMY_LABELS"]
MODALITY_MAP = meta["MODALITY_MAP"]
MED_CONFIG = meta["MED_CONFIG"]
TOTAL_DISEASE_DIM = meta["TOTAL_DISEASE_DIM"]

In [12]:
import torch
import medmnist
from medmnist import INFO
from torch.utils.data import Dataset, ConcatDataset
from torchvision import transforms, models

# 1. Load your saved metadata
with open(f"/kaggle/input/metadata/navigator_metadata.pkl", "rb") as f:
    meta = pickle.load(f)

ANATOMY_LABELS = meta["ANATOMY_LABELS"]
MODALITY_MAP = meta["MODALITY_MAP"]
MED_CONFIG = meta["MED_CONFIG"]
TOTAL_DISEASE_DIM = meta["TOTAL_DISEASE_DIM"]

# 2. Updated Dataset Wrapper with Error Handling
class UnifiedMedMNIST(Dataset):
    def __init__(self, name, cfg):
        DataClass = getattr(medmnist, INFO[name]["python_class"])
        # download=True will use the files we moved to Google Drive
        self.ds = DataClass(split="train", download=True, size=28)
        self.cfg = cfg
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        img, target = self.ds[idx]
        img = self.transform(img.convert("RGB"))
        target = target.flatten()
        disease_vec = torch.zeros(TOTAL_DISEASE_DIM)
        for i, val in enumerate(target):
            idx_val = int(val) if len(target) == 1 else i
            if len(target) == 1 or val == 1:
                if (self.cfg["off"] + idx_val) < TOTAL_DISEASE_DIM:
                    disease_vec[self.cfg["off"] + idx_val] = 1.0
        return {
            "image": img, 
            "anatomy": ANATOMY_LABELS[self.cfg["anatomy"]],
            "modality": MODALITY_MAP[self.cfg["modality"]], 
            "disease_label": disease_vec
        }

# 3. Safe Reconstruction Loop
datasets = []
available_medminsts = set(medmnist.INFO.keys())

print("Reconstructing Master Dataset...")
for name, cfg in MED_CONFIG.items():
    if name in available_medminsts:
        try:
            datasets.append(UnifiedMedMNIST(name, cfg))
            print(f"✓ Successfully loaded: {name}")
        except Exception as e:
            print(f"✗ Failed to load {name}: {e}")
    else:
        print(f"⚠ Skipping {name}: Not found in this MedMNIST version.")

Reconstructing Master Dataset...
✓ Successfully loaded: pathmnist
✓ Successfully loaded: breastmnist
✓ Successfully loaded: retinamnist
✓ Successfully loaded: chestmnist
⚠ Skipping fracturemnist: Not found in this MedMNIST version.
⚠ Skipping tpmmnist: Not found in this MedMNIST version.


100%|██████████| 19.7M/19.7M [00:21<00:00, 913kB/s] 


✓ Successfully loaded: dermamnist


100%|██████████| 35.5M/35.5M [00:26<00:00, 1.36MB/s]


✓ Successfully loaded: bloodmnist


100%|██████████| 38.2M/38.2M [00:47<00:00, 806kB/s] 


✓ Successfully loaded: organamnist


In [21]:
class GlobalMedicalNavigator(nn.Module):
    def __init__(self, num_anatomy=10, num_modality=7, embedding_dim=512):
        super(GlobalMedicalNavigator, self).__init__()

        # Shared Visual Backbone
        backbone = models.densenet121(weights=None) # No need for ImageNet weights, loading ours
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = 1024

        # Embedding Head
        self.embedding_head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, embedding_dim),
            nn.LayerNorm(embedding_dim) 
        )

        # Anatomy Head
        self.anatomy_head = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_anatomy)
        )

        # Modality Head
        self.modality_head = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_modality)
        )

    def forward(self, x):
        feature_maps = self.features(x)
        pooled = self.pool(feature_maps)
        flattened = torch.flatten(pooled, 1)

        z = self.embedding_head(flattened)
        anatomy_logits = self.anatomy_head(flattened)
        modality_logits = self.modality_head(flattened)

        return {
            "z": z,                      
            "anatomy_probs": anatomy_logits, 
            "modality_probs": modality_logits, 
            "feature_maps": feature_maps     
        }


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
master_ds = ConcatDataset(datasets)

In [23]:
model = GlobalMedicalNavigator(num_anatomy=10, num_modality=len(MODALITY_MAP)).to(device)
model.load_state_dict(torch.load("/kaggle/input/navigator/pytorch/default/1/global_navigator.pth", map_location=device))
model.eval()

GlobalMedicalNavigator(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [27]:
import faiss
import torch
import numpy as np
import pickle
from tqdm import tqdm

# ==========================================
# 1. THE IMPROVED RETRIEVAL ENGINE
# ==========================================
class ImprovedMedicalRetrieval:
    def __init__(self, embedding_dim=512):
        # We use IndexFlatL2 for high-precision exact search
        self.index = faiss.IndexFlatL2(embedding_dim)
        self.metadata = [] 

    def add_to_memory(self, z_vectors, metadata_list):
        # Convert to float32 numpy for FAISS
        vectors = z_vectors.detach().cpu().numpy().astype('float32')
        self.index.add(vectors)
        self.metadata.extend(metadata_list)

    def query(self, query_z, query_anatomy=None, k=5):
        """
        Your Improved Query Logic: 
        1. Oversamples (K*5)
        2. Filters by the Top-Down Anatomy Guess
        """
        query_vec = query_z.detach().cpu().numpy().astype('float32')
        
        # Search for more than K neighbors so we can filter by anatomy
        distances, indices = self.index.search(query_vec, k * 5)
        
        results, filtered_distances = [], []
        # Process each item in the batch
        for d_batch, i_batch in zip(distances, indices):
            item_results = []
            item_dists = []
            for dist, idx in zip(d_batch, i_batch):
                meta = self.metadata[idx]
                
                # IMPROVEMENT: Filter by Predicted Anatomy
                if query_anatomy is None or meta['anatomy'] == query_anatomy:
                    item_results.append(meta)
                    item_dists.append(dist)
                
                if len(item_results) >= k:
                    break
            
            results.append(item_results)
            filtered_distances.append(item_dists)
            
        return results, filtered_distances

    def weighted_disease_average(self, neighbors, distance_list):
        """
        IMPROVEMENT: Calculates the Disease Prior 
        Weighted by visual similarity (1/distance)
        """
        disease_vectors = []
        weights = []
        
        for meta, dist in zip(neighbors, distance_list):
            d_vec = meta['disease_label']
            # Ensure it is a numpy array
            if torch.is_tensor(d_vec): d_vec = d_vec.numpy()
            
            # Masking Logic: Treat any -1.0 as 0.0 for the average
            # (In your current dataset these are 0.0 anyway)
            d_vec_clean = np.where(d_vec == -1.0, 0.0, d_vec)
            
            disease_vectors.append(d_vec_clean)
            # Similarity weight (inverse distance)
            weights.append(1 / (dist + 1e-6))
        
        disease_vectors = np.stack(disease_vectors)
        weights = np.array(weights).reshape(-1, 1)
        
        # Calculate Weighted Average
        weighted_avg = np.sum(disease_vectors * weights, axis=0) / np.sum(weights)
        return torch.tensor(weighted_avg, dtype=torch.float32)

# ==========================================
# 2. POPULATION SCRIPT (INDEXING)
# ==========================================
def build_visual_memory(model, dataset, device):
    model.eval()
    engine = ImprovedMedicalRetrieval(embedding_dim=512)
    
    # DataLoader for indexing (Batch size 64)
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    
    print("\n--- Building Improved Visual Memory (90k+ Cases) ---")
    with torch.no_grad():
        for batch in tqdm(loader):
            images = batch['image'].to(device)
            
            # Forward pass through the backbone you just trained
            outputs = model(images)
            z = outputs['z']
            
            # Extract metadata from batch
            batch_metadata = []
            for i in range(images.size(0)):
                batch_metadata.append({
                    'anatomy': batch['anatomy'][i].item(),
                    'modality': batch['modality'][i].item(),
                    'disease_label': batch['disease_label'][i].cpu() # Keep on CPU
                })
            
            engine.add_to_memory(z, batch_metadata)
            
    return engine

# ==========================================
# 3. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Initialize and Load your saved Navigator
    # Use the model class from your training code
    model = GlobalMedicalNavigator(num_anatomy=10, num_modality=7).to(device)
    model.load_state_dict(torch.load("/kaggle/input/navigator/pytorch/default/1/global_navigator.pth"))
    
    # 2. Build the engine
    visual_memory = build_visual_memory(model, master_ds, device)
    
    # 3. Persistent Save
    faiss.write_index(visual_memory.index, "medical_memory.index")
    with open("memory_metadata.pkl", "wb") as f:
        pickle.dump(visual_memory.metadata, f)
        
    print(f"\nSUCCESS: Engine built with {visual_memory.index.ntotal} cases.")


--- Building Improved Visual Memory (90k+ Cases) ---


100%|██████████| 3495/3495 [12:20<00:00,  4.72it/s]



SUCCESS: Engine built with 223617 cases.
